<a href="https://colab.research.google.com/github/hquinnett/final-project-GB885-quinnett-h/blob/main/GB885_Final_Project_Draft.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# RUSH Sales Analysis

Hailey Quinnett  
GB 885 - Python Fundamentals  
Final Project  
8/9/2026

## Overview

RUSH is a globally renowned sportswear and footwear brand known for its innovative designs and performance-oriented products. This project analyzes RUSH's raw US sales data from 2020 and 2021 to identify trends and insights that can help company leadership understand the market and identify opportunities for growth.

The company stores its raw sales data as a collection of three tables:

*   TABLE_PRODUCTS
*   TABLE_RETAILER
*   TABLE_SALES

The data includes the number of units sold, the total sales revenue, the location of the sales, the type of product sold, as well as other relevant information.

## Business Questions

The VP of US Sales asked for answers to the following questions:

1.   What product category (product) had the highest sales (in dollars) in 2021? How much did it sell?
2.   What state had the highest sales (in dollars) of women's products in 2021? How much was it?
3.   What state had the highest sales (in dollars) of men's products in 2021? How much was it?
4.   What retailer purchased the most units in 2021? In 2020?





## Import libraries & datasets

In [ ]:
# Import libraries
import pandas as pd
import sklearn

In [ ]:
# Load data from github repository
df_sales = pd.read_csv('https://raw.githubusercontent.com/hquinnett/final-project-GB885-quinnett-h/refs/heads/main/data/TABLE_SALES_885.csv')
df_retailer = pd.read_csv('https://raw.githubusercontent.com/hquinnett/final-project-GB885-quinnett-h/refs/heads/main/data/TABLE_RETAILER_885.csv')
# Set delimiter to '|' for products csv
df_products = pd.read_csv('https://raw.githubusercontent.com/hquinnett/final-project-GB885-quinnett-h/refs/heads/main/data/TABLE_PRODUCTS_885.csv', delimiter='|')

In [ ]:
# Preview sales data
df_sales.head()

In [ ]:
# Preview retailer data
df_retailer.head()

In [ ]:
# Preview products data
df_products.head()

In [ ]:
# Merge retailer and sales dataframes
df = df_sales.merge(df_retailer, on='RETAILER_ID', how='left')

# Merge product dataframe into the main dataframe
df= df.merge(df_products, on= 'PRODUCT_ID', how='left')

# Preview merged data
df.head()

## Data inspection

In [ ]:
df.info()

In [ ]:
# Features not needed for the analysis: ORDER_ID, RETAILER_ID, PRODUCT_ID
df = df.drop(['ORDER_ID', 'RETAILER_ID', 'PRODUCT_ID'], axis=1)

df.info()

In [ ]:
# UNITS_SOLD has data type object - check for nonint values

# Inspect unique values
print(df['UNITS_SOLD'].unique())

In [ ]:
# Check rows containing '***' for UNITS_SOLD
df[df['UNITS_SOLD'] == '***']

In [ ]:
# Change '***' for UNITS_SOLD to null
df['UNITS_SOLD'] = df['UNITS_SOLD'].replace('***', pd.NA)

# check
df[df['UNITS_SOLD'] == '***']

In [ ]:
# Convert UNITS_SOLD column to numeric
df['UNITS_SOLD'] = pd.to_numeric(df['UNITS_SOLD'])

# Check
df['UNITS_SOLD'].dtype

In [ ]:
# INVOICE_DATE has data type object

# Inspect unique values
print(df['INVOICE_DATE'].unique())

In [ ]:
# Convert INVOICE_DATE to datetime
df['INVOICE_DATE'] = pd.to_datetime(df['INVOICE_DATE'])

# Check
df['INVOICE_DATE'].dtype

In [ ]:
# Inspect for traditional null values
df.isnull().sum()

In [ ]:
# Inspect for non traditional missing values - categorical

# list of categorical variables
cat_var = list(df.select_dtypes(include=['object']).columns)

# view unique values for each of those variables
for column in cat_var:
  print(column)
  print(df[column].unique())

In [ ]:
# Inspect for non traditional missing values - numerical (99999)
df.describe()

In [ ]:
# Inspect for duplicates
df.duplicated().sum()

In [ ]:
# List duplicated rows
df[df.duplicated()]

In [ ]:
# Inspect for erroneous values - numerical
df.describe()

In [ ]:
# Inspect for erroneous values - categorical
cat_var = list(df.select_dtypes(include=['object']).columns)

for column in cat_var:
  print(column)
  print(df[[column]].value_counts())

In [ ]:
# Use IQR to check for outliers
def count_iqr_outliers(df, column):
    # define q1
    q1 = df[column].quantile(0.25)
    # define q3
    q3 = df[column].quantile(0.75)
    # define iqr
    iqr = q3 - q1
    # define outlier thresholds
    l_threshold = q1 - 1.5 * iqr
    u_threshold = q3 + 1.5 * iqr
    # dount outliers
    outliers = (df[column] < l_threshold) | (df[column] > u_threshold)
    # Count the number of True values (outliers)
    return outliers.sum()

num_var = list(df.select_dtypes(include=['int64', 'float64']).columns)

for column in num_var:
  print(f'{column} : {count_iqr_outliers(df, column)}')

### Clean data

In [ ]:
# Examine UNITS_SOLD missing values
df[df['UNITS_SOLD'].isnull()]

In [ ]:
# Remove rows with missing units sold
df = df.dropna(subset=['UNITS_SOLD'])
df['UNITS_SOLD'].isnull().sum()

In [ ]:
# Examine PRICE_PER_UNIT missing values
df[df['PRICE_PER_UNIT'].isnull()]

In [ ]:
# Remove rows with missing price per unit
df = df.dropna(subset=['PRICE_PER_UNIT'])
df['PRICE_PER_UNIT'].isnull().sum()

In [ ]:
# Examine PRICE_PER_UNIT values of 99999
(df['PRICE_PER_UNIT'] == 99999).sum()

In [ ]:
# Remove row with 99999 PRICE_PER_UNIT
df = df[df['PRICE_PER_UNIT'] != 99999]
(df['PRICE_PER_UNIT'] == 99999).sum()

In [ ]:
# Examine rows with missing retailer information
df[
    df['RETAILER'].isnull() |
    df['REGION'].isnull() |
    df['STATE'].isnull() |
    df['CITY'].isnull()
]

In [ ]:
# Determine retailer info from sales dataframe
df_sales.loc[1534]

In [ ]:
# PUll missing retailer info from retailer dataframe
df_retailer[df_retailer['RETAILER_ID'] == 'W00WUTSA']

In [ ]:
# Add recovered retailer info to main dataframe
df.loc[1534, ['RETAILER', 'REGION', 'STATE', 'CITY']] = ['West Gear', 'West', 'Utah', 'Salt Lake City']

In [ ]:
# Drop duplicate Values
df = df.drop_duplicates()

In [ ]:
# Replace erroneous value "Ootlet" with "Outlet"
df.replace(to_replace='Ootlet', value = "Outlet", inplace = True)

In [ ]:
#re-evaluate outliers post cleaning
#list of numerical variables
num_var = list(df.select_dtypes(include=['int64', 'float64']).columns)

for column in num_var:
    print(column)
    print(count_iqr_outliers(df, column))

In [ ]:
# Check year values
df['YEAR'].value_counts()

In [ ]:
# Windsorize price per unit, units sold, operating margin
wind_var = ['PRICE_PER_UNIT', 'UNITS_SOLD', 'OPERATING_MARGIN']

for column in wind_var:
  #set upper clipping threshold
  high_percentile_value = df[column].quantile(0.95)
  #clip upper outliers
  df.loc[:,column] = df[column].clip(upper=high_percentile_value)
  #set clipping threshold
  low_percentile_value = df[column].quantile(0.05)
  #clip outliers
  df.loc[:,column] = df[column].clip(lower=low_percentile_value)

#check work
for column in wind_var:
    print(column)
    print(count_iqr_outliers(df, column))

### Business Questions

In [ ]:
# Create a total sales field
df['TOTAL_SALES'] = df['PRICE_PER_UNIT'] * df['UNITS_SOLD']

# Separate data by year
df_2020 = df[df['YEAR'] == 2020]
df_2021 = df[df['YEAR'] == 2021]

In [ ]:
# Calculate sales for each product category in 2021
df_2021.groupby('PRODUCT_NAME')['TOTAL_SALES'].sum().sort_values()

In [ ]:
# Calculate sales for women's products for each state in 2021
df_2021[df_2021['PRODUCT_NAME'].str.contains('''Women's''')].groupby('STATE')['TOTAL_SALES'].sum().sort_values(ascending =False)

In [ ]:
# Calculate sales for men's products for each state in 2021
df_2021[df_2021['PRODUCT_NAME'].str.contains('''Men's''')].groupby('STATE')['TOTAL_SALES'].sum().sort_values(ascending =False)

In [ ]:
# Calculate how many units each retailer purchased in 2021
df_2021.groupby('RETAILER')['UNITS_SOLD'].sum().sort_values()

In [ ]:
# Calculate how many units each retailer purchased in 2020
df_2020.groupby('RETAILER')['UNITS_SOLD'].sum().sort_values()

### Additional Insights

In [ ]:
# Calculate total sales for each retailer in 2021
df_2021.groupby('RETAILER')['TOTAL_SALES'].sum().sort_values()

In [ ]:
# Calculate total sales for each retailer in 2020
df_2020.groupby('RETAILER')['TOTAL_SALES'].sum().sort_values()

In [ ]:
# Calculate sales for each sales method in 2021
df_2021.groupby('SALES_METHOD')['TOTAL_SALES'].sum().sort_values()

In [ ]:
# Calculate sales for each sales method in 2020
df_2020.groupby('SALES_METHOD')['TOTAL_SALES'].sum().sort_values()

In [ ]:
# Calculate sales in 2021 broken down by retailer and sales medthod
df_2021.groupby(['RETAILER', 'SALES_METHOD'])['TOTAL_SALES'].sum()

In [ ]:
# Calculate sales by month
df.groupby(['YEAR', 'MONTH'])['TOTAL_SALES'].sum()